# FIT Training Data Analysis

This notebook parses the `.fit` files in `../_data/_training_data`, builds an activity-level summary table, and gives you a few starting points for bespoke analysis.

The goal is to make it easy to answer questions like:
- How many runs, swims, and rides do I have?
- What does my weekly volume look like?
- Which workouts were longest or hardest?
- What do the lap- or record-level messages look like for a given activity?


In [ ]:
from pathlib import Path

import fitdecode
import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 140)

DATA_DIR = Path("../_data/_training_data")


In [ ]:
def message_to_dict(frame):
    row = {}
    for field in frame.fields:
        row[field.name] = field.value
    return row


def detect_source(path: Path) -> str:
    name = path.name.lower()
    if "zwift" in name:
        return "zwift"
    if "garmin" in name:
        return "garmin"
    return "other"


def parse_activity_file(path: Path) -> dict:
    file_id = {}
    session = {}

    with fitdecode.FitReader(path) as fit:
        for frame in fit:
            if not isinstance(frame, fitdecode.FitDataMessage):
                continue

            if frame.name == "file_id" and not file_id:
                file_id = message_to_dict(frame)
            elif frame.name == "session" and not session:
                session = message_to_dict(frame)

            if file_id and session:
                break

    avg_speed = session.get("avg_speed")
    max_speed = session.get("max_speed")

    return {
        "file_path": str(path),
        "file_name": path.name,
        "month_folder": path.parent.name,
        "source": detect_source(path),
        "manufacturer": file_id.get("manufacturer"),
        "device": file_id.get("garmin_product") or file_id.get("product"),
        "sport": session.get("sport"),
        "sub_sport": session.get("sub_sport"),
        "start_time": session.get("start_time"),
        "elapsed_hours": (session.get("total_elapsed_time") or 0) / 3600,
        "moving_hours": (session.get("total_timer_time") or 0) / 3600,
        "distance_km": (session.get("total_distance") or 0) / 1000,
        "calories": session.get("total_calories"),
        "avg_hr": session.get("avg_heart_rate"),
        "max_hr": session.get("max_heart_rate"),
        "avg_power": session.get("avg_power"),
        "max_power": session.get("max_power"),
        "avg_cadence": session.get("avg_cadence"),
        "avg_speed_kmh": avg_speed * 3.6 if avg_speed is not None else None,
        "max_speed_kmh": max_speed * 3.6 if max_speed is not None else None,
        "pool_length_m": session.get("pool_length"),
        "num_active_lengths": session.get("num_active_lengths"),
    }


def build_activity_table(data_dir: Path = DATA_DIR) -> pd.DataFrame:
    files = sorted(
        path for path in data_dir.rglob("*") if path.is_file() and path.suffix.lower() == ".fit"
    )
    rows = [parse_activity_file(path) for path in files]
    df = pd.DataFrame(rows).sort_values("start_time").reset_index(drop=True)

    start_ts = pd.to_datetime(df["start_time"]).dt.tz_localize(None)
    df["activity_date"] = start_ts.dt.date
    df["week"] = start_ts.dt.to_period("W").astype(str)
    df["month"] = start_ts.dt.to_period("M").astype(str)
    return df


def extract_messages(path: Path, message_name: str) -> pd.DataFrame:
    rows = []
    with fitdecode.FitReader(path) as fit:
        for frame in fit:
            if isinstance(frame, fitdecode.FitDataMessage) and frame.name == message_name:
                rows.append(message_to_dict(frame))
    return pd.DataFrame(rows)


In [ ]:
activity_df = build_activity_table()
print(f"Parsed {len(activity_df)} FIT files")
activity_df.head(10)


In [ ]:
sport_summary = (
    activity_df.groupby(["sport", "sub_sport"], dropna=False)
    .agg(
        workouts=("file_name", "count"),
        total_distance_km=("distance_km", "sum"),
        total_moving_hours=("moving_hours", "sum"),
        avg_calories=("calories", "mean"),
    )
    .sort_values(["workouts", "total_distance_km"], ascending=False)
)

source_summary = (
    activity_df.groupby("source")
    .agg(
        workouts=("file_name", "count"),
        total_distance_km=("distance_km", "sum"),
        total_moving_hours=("moving_hours", "sum"),
    )
    .sort_values("workouts", ascending=False)
)

sport_summary


In [ ]:
source_summary


In [ ]:
weekly_volume = (
    activity_df.groupby(["week", "sport"], dropna=False)
    .agg(distance_km=("distance_km", "sum"), moving_hours=("moving_hours", "sum"))
    .reset_index()
)

weekly_distance = weekly_volume.pivot(index="week", columns="sport", values="distance_km").fillna(0)
weekly_hours = weekly_volume.pivot(index="week", columns="sport", values="moving_hours").fillna(0)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
weekly_distance.plot(kind="bar", stacked=True, ax=axes[0], title="Weekly Distance by Sport")
weekly_hours.plot(kind="bar", stacked=True, ax=axes[1], title="Weekly Moving Hours by Sport")
axes[0].set_ylabel("km")
axes[1].set_ylabel("hours")
axes[1].set_xlabel("week")
plt.tight_layout()


In [ ]:
# Example bespoke queries

# Longest activities overall
activity_df.sort_values("distance_km", ascending=False)[
    ["activity_date", "sport", "sub_sport", "distance_km", "moving_hours", "avg_power", "avg_hr", "file_name"]
].head(10)


In [ ]:
# Inspect one file in detail by changing TARGET_FILE
TARGET_FILE = activity_df.iloc[0]["file_path"]

lap_df = extract_messages(Path(TARGET_FILE), "lap")
record_df = extract_messages(Path(TARGET_FILE), "record")

print("Target file:", TARGET_FILE)
print("lap rows:", len(lap_df))
print("record rows:", len(record_df))

lap_df.head(), record_df.head()


## Ideas for next analyses

- Compare running pace or cycling power by week.
- Track swim volume and average heart rate over time.
- Isolate treadmill runs vs. outdoor runs.
- Build a merged chart of hours across all sports.
- Export `activity_df` to CSV for use in the website or in other tools.

A simple export cell would be:

```python
activity_df.to_csv("../_data/training_activity_summary.csv", index=False)
```
